In [1]:
import torch
from diffusion.flows.prob_paths import GaussianCondProbPath
from diffusion.training.trainer_flow import FlowTrainer
from diffusion.sampleables.sampleable_mnist import MNISTSampleable
from diffusion.backbones.res_unet_attn import ResUnet

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [3]:
sampeable = MNISTSampleable(train=True)
val_sampeable = MNISTSampleable(train=False)

path = GaussianCondProbPath(p_data=sampeable, p_simple_shape=sampeable.shape).to(device)

val_path = GaussianCondProbPath(
    p_data=val_sampeable, p_simple_shape=sampeable.shape
).to(device)

backbone = ResUnet(
    in_channels=1,
    channel_dims=[16, 32, 64],
    use_attention=[False, False, True],
    num_classes=sampeable.num_classes,
    t_dim=64,
    y_dim=32,
    cond_dim=64,
).to(device)

trainer = FlowTrainer(
    path=path,
    val_path=val_path,
    backbone=backbone,
    null_class=sampeable.num_classes,
)

In [4]:
state_dict = trainer.train(
    num_epochs=15,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=500,
    validate=True,
    plot_path="./loss.png",
)

2026-01-06 00:37:40,037 - flow-matching - INFO - Training model with size: 2.485 MiB
Epoch 0/15: 100%|██████████| 500/500 [00:46<00:00, 10.74it/s, train_loss=0.247952]
2026-01-06 00:38:26,676 - flow-matching - INFO - val loss: 0.1705177277326584, best val loss: 0.1705177277326584
Epoch 1/15: 100%|██████████| 500/500 [00:46<00:00, 10.80it/s, train_loss=0.162095]
2026-01-06 00:39:13,155 - flow-matching - INFO - val loss: 0.16280432045459747, best val loss: 0.16280432045459747
Epoch 2/15: 100%|██████████| 500/500 [00:46<00:00, 10.68it/s, train_loss=0.149466]
2026-01-06 00:40:00,115 - flow-matching - INFO - val loss: 0.1382865309715271, best val loss: 0.1382865309715271
Epoch 3/15: 100%|██████████| 500/500 [00:46<00:00, 10.86it/s, train_loss=0.144021]
2026-01-06 00:40:46,294 - flow-matching - INFO - val loss: 0.15401431918144226, best val loss: 0.1382865309715271
Epoch 4/15: 100%|██████████| 500/500 [00:46<00:00, 10.72it/s, train_loss=0.138871]
2026-01-06 00:41:33,055 - flow-matching - INF

In [7]:
torch.save(state_dict, "./models/backbone_flow.pt")